# **Import Essential Packages**
**Name**: B01_zero_shot.ipynb

**Purpose**: In this notebook, I build a simple baseline that integretates LLM power as the Head of the framework to understanding questions an generate answers.

**Author**: Khang Phuc Nguyen

**Date**: 09-05-2026

In [ ]:
!python -m pip install --upgrade pip
!pip install -q pandas tqdm
!pip install -q transformers torch 

# **1. Essential Imports and Configuration**

This section prepares everything that the zero-shot baseline needs before we call any model:

- locate the project root reliably;
- define paths to the organizer datasets;
- define where predictions/reports will be saved;
- configure reproducibility and model settings;
- keep the notebook easy to run in both **mock mode** and **real local LLM mode**.

For the first baseline, we want the setup to be simple and explicit because later experiments will compare against this zero-shot result.

In [16]:
# Python standard library imports.
# These modules are enough for path handling, JSON files, environment variables,
# reproducibility, simple timestamped artifact names, and local HTTP calls.
import json
import os
import random
import sys
import urllib.error
import urllib.request
from datetime import datetime
from pathlib import Path

# Third-party imports.
# pandas is used for tabular dataset handling.
# tqdm gives progress bars when we later run the model over many examples.
import pandas as pd
from tqdm.auto import tqdm


# -----------------------------------------------------------------------------
# 1. Locate the repository root
# -----------------------------------------------------------------------------
# A notebook can be launched from different working directories depending on
# whether you open it from VS Code, JupyterLab, or the terminal. Instead of
# assuming the current directory is the project root, we walk upward until we
# find pyproject.toml, which identifies the Exact2026 repository.
def find_repo_root(start: Path | None = None) -> Path:
    """Return the Exact2026 repository root by searching for pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        "Could not find pyproject.toml. Please run this notebook inside the Exact2026 repository."
    )


ROOT = find_repo_root()
SRC_DIR = ROOT / "src"

# Add src/ to Python's import path so future project modules can be imported as
# normal packages. This is useful once we implement reusable code outside the
# notebook, for example src/baselines/llm_client.py or evaluation utilities.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# -----------------------------------------------------------------------------
# 2. Dataset paths
# -----------------------------------------------------------------------------
# The organizer released two text-only datasets:
# - Type 1: logic / education regulation questions with natural-language premises.
# - Type 2: physics questions with CoT, answer, and unit fields.
DATA_DIR = SRC_DIR / "data"
TYPE1_PATH = (
    DATA_DIR
    / "Logic_Based_Educational_Queries_Text_Only"
    / "Logic_Based_Educational_Queries.json"
)
TYPE2_PATH = DATA_DIR / "Physics_Problems_Text_Only.csv"

# Check early that the expected files exist. Failing here is easier to debug than
# failing later inside a loading/evaluation loop.
assert TYPE1_PATH.is_file(), f"Type 1 dataset not found: {TYPE1_PATH}"
assert TYPE2_PATH.is_file(), f"Type 2 dataset not found: {TYPE2_PATH}"


# -----------------------------------------------------------------------------
# 3. Artifact paths
# -----------------------------------------------------------------------------
# All outputs from this baseline should go to artifacts/ so that:
# - notebooks stay clean;
# - predictions can be inspected later;
# - metrics can be reused in the paper/report;
# - future baselines can be compared fairly.
ARTIFACTS_DIR = ROOT / "artifacts"
PREDICTIONS_DIR = ARTIFACTS_DIR / "predictions"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
SPLITS_DIR = ARTIFACTS_DIR / "splits"

for directory in [ARTIFACTS_DIR, PREDICTIONS_DIR, REPORTS_DIR, SPLITS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# A run name makes saved files easy to identify. We keep it deterministic enough
# to read but unique enough that multiple experiments do not overwrite each other.
RUN_NAME = "B01_zero_shot"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = f"{RUN_NAME}_{RUN_TIMESTAMP}"


# -----------------------------------------------------------------------------
# 4. Reproducibility settings
# -----------------------------------------------------------------------------
# Zero-shot LLM outputs may still vary depending on sampling settings. We set a
# seed now so that any random splits, sampling, or mock outputs are repeatable.
SEED = 42
random.seed(SEED)

# Optional split file. If it does not exist yet, later cells can create it or run
# on the full dataset. Keeping the path here makes future evaluation consistent.
SPLIT_FILE = SPLITS_DIR / f"seed{SEED}.json"


# -----------------------------------------------------------------------------
# 5. Model and inference settings
# -----------------------------------------------------------------------------
# The challenge restricts submitted systems to open-source/open-weight LLMs with
# at most 8B parameters. This notebook supports:
# - mock mode for fast pipeline testing;
# - Ollama mode for a local quantized model without torch/transformers;
# - Hugging Face mode for direct transformers inference when available.
#
# Example terminal usage:
#   MOCK_LLM=1 jupyter notebook
#   EXACT_LLM_BACKEND=ollama OLLAMA_MODEL_ID="qwen2.5:7b" jupyter notebook
#   EXACT_LLM_BACKEND=huggingface EXACT_MODEL_ID="Qwen/Qwen2.5-7B-Instruct" jupyter notebook
MODEL_ID = os.getenv("EXACT_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")
OLLAMA_MODEL_ID = os.getenv("OLLAMA_MODEL_ID", "qwen2.5:7b")
OLLAMA_GENERATE_URL = os.getenv("OLLAMA_GENERATE_URL", "http://127.0.0.1:11434/api/generate")

# MOCK_LLM=1 lets us test data loading, prompt building, parsing, and evaluation
# without downloading or running a real model. This is useful for quick debugging.
MOCK_LLM = os.getenv("MOCK_LLM", "0") == "1"

# Ollama is the default real backend because it works locally with the downloaded
# qwen2.5:7b model and does not require installing torch/transformers.
LLM_BACKEND = os.getenv("EXACT_LLM_BACKEND", "ollama")

# Conservative generation settings for a baseline. Low temperature reduces
# randomness, which makes evaluation and error analysis easier.
MAX_NEW_TOKENS = int(os.getenv("EXACT_MAX_NEW_TOKENS", "512"))
TEMPERATURE = float(os.getenv("EXACT_TEMPERATURE", "0.0"))
TOP_P = float(os.getenv("EXACT_TOP_P", "1.0"))


# -----------------------------------------------------------------------------
# 6. Small preview of the configuration
# -----------------------------------------------------------------------------
print("Exact2026 zero-shot baseline configuration")
print(f"  Repository root : {ROOT}")
print(f"  Type 1 dataset  : {TYPE1_PATH}")
print(f"  Type 2 dataset  : {TYPE2_PATH}")
print(f"  Artifacts dir   : {ARTIFACTS_DIR}")
print(f"  Run ID          : {RUN_ID}")
print(f"  Seed            : {SEED}")
print(f"  Mock LLM        : {MOCK_LLM}")
print(f"  LLM backend     : {LLM_BACKEND}")
print(f"  HF model ID     : {MODEL_ID}")
print(f"  Ollama model ID : {OLLAMA_MODEL_ID}")
print(f"  Ollama URL      : {OLLAMA_GENERATE_URL}")
print(f"  Max new tokens  : {MAX_NEW_TOKENS}")
print(f"  Temperature     : {TEMPERATURE}")
print(f"  Top-p           : {TOP_P}")

Exact2026 zero-shot baseline configuration
  Repository root : /home/phuckhang/MyWorkspace/Exact2026
  Type 1 dataset  : /home/phuckhang/MyWorkspace/Exact2026/src/data/Logic_Based_Educational_Queries_Text_Only/Logic_Based_Educational_Queries.json
  Type 2 dataset  : /home/phuckhang/MyWorkspace/Exact2026/src/data/Physics_Problems_Text_Only.csv
  Artifacts dir   : /home/phuckhang/MyWorkspace/Exact2026/artifacts
  Run ID          : B01_zero_shot_20260512_211834
  Seed            : 42
  Mock LLM        : False
  LLM backend     : ollama
  HF model ID     : Qwen/Qwen2.5-7B-Instruct
  Ollama model ID : qwen2.5:7b
  Ollama URL      : http://127.0.0.1:11434/api/generate
  Max new tokens  : 512
  Temperature     : 0.0
  Top-p           : 1.0


# **2. Data Loading and Normalization**

The organizer released two datasets with different structures:

1. **Type 1 / Logic** is a JSON list. Each item contains one premise set and one or more questions about that same premise set.
2. **Type 2 / Physics** is a CSV file. Each row is one physics problem.

For the zero-shot baseline, we normalize both datasets into one row-per-question table. This makes later prompting and evaluation much easier because every row has the same core fields:

- `id`: unique example id;
- `task_type`: `logic` or `physics`;
- `question`: the question to answer;
- `gold_answer`: organizer-provided answer;
- `gold_explanation`: organizer explanation or CoT;
- task-specific metadata such as premises, FOL, unit, and ID prefix.

In [17]:
def detect_logic_question_type(question: str, answer: str) -> str:
    """Classify Type 1 questions into broad answer-format groups."""
    question = question or ""
    answer = str(answer or "").strip()

    # Multiple-choice questions in this dataset usually include options A-D inside
    # the question text. Their gold answer is normally one of A, B, C, or D.
    if "\nA." in question or answer in {"A", "B", "C", "D"}:
        return "multiple_choice"

    # The official task commonly uses Yes / No / Unknown labels for entailment.
    if answer in {"Yes", "No", "Unknown", "Uncertain", "False"}:
        return "yes_no_unknown"

    return "other"


def detect_physics_answer_type(answer: str, unit: str) -> str:
    """Classify Type 2 answers by whether they look numeric, conceptual, or missing."""
    answer = str(answer or "").strip()
    unit = str(unit or "").strip()

    if not answer:
        return "missing_answer"

    # This intentionally stays simple. Later we can improve numeric parsing for
    # fractions, scientific notation, intervals, and vector answers.
    has_digit = any(ch.isdigit() for ch in answer)
    if has_digit:
        return "numeric_with_unit" if unit and unit not in {"-", "—"} else "numeric_no_unit"

    return "conceptual"


def physics_id_prefix(example_id: str) -> str:
    """Return the alphabetic prefix of a physics ID, e.g. TD401 -> TD."""
    prefix = ""
    for ch in str(example_id):
        if ch.isalpha():
            prefix += ch
        else:
            break
    return prefix or "unknown"


def load_logic_dataset(path: Path) -> pd.DataFrame:
    """Load and flatten the Type 1 logic dataset into one row per question."""
    raw_records = json.loads(path.read_text(encoding="utf-8"))
    rows = []

    for group_index, record in enumerate(raw_records):
        premises_nl = record.get("premises-NL", [])
        premises_fol = record.get("premises-FOL", [])
        support_indices = record.get("idx", [])
        questions = record.get("questions", [])
        answers = record.get("answers", [])
        explanations = record.get("explanation", [])

        # Each original record can contain multiple questions over the same
        # premise set. We create one normalized row per question.
        for question_index, question in enumerate(questions):
            answer = answers[question_index] if question_index < len(answers) else ""
            explanation = explanations[question_index] if question_index < len(explanations) else ""
            support_idx = support_indices[question_index] if question_index < len(support_indices) else []
            question_type = detect_logic_question_type(question, answer)

            rows.append(
                {
                    "id": f"logic_{group_index:04d}_{question_index:02d}",
                    "group_id": f"logic_{group_index:04d}",
                    "task_type": "logic",
                    "question_type": question_type,
                    "question": question,
                    "premises_nl": premises_nl,
                    "premises_fol": premises_fol,
                    "support_idx": support_idx,
                    "gold_answer": str(answer).strip(),
                    "gold_unit": "",
                    "gold_explanation": explanation,
                    "source_path": str(path),
                    # This label is used only for split balancing.
                    "stratify_label": f"logic::{question_type}",
                }
            )

    return pd.DataFrame(rows)


def load_physics_dataset(path: Path) -> pd.DataFrame:
    """Load the Type 2 physics CSV dataset into the normalized row format."""
    raw_df = pd.read_csv(path).fillna("")
    rows = []

    for _, record in raw_df.iterrows():
        example_id = str(record.get("id", "")).strip()
        answer = str(record.get("answer", "")).strip()
        unit = str(record.get("unit", "")).strip()
        prefix = physics_id_prefix(example_id)
        answer_type = detect_physics_answer_type(answer, unit)

        rows.append(
            {
                "id": f"physics_{example_id}",
                "group_id": f"physics_{example_id}",
                "task_type": "physics",
                "question_type": "physics",
                "question": str(record.get("question", "")).strip(),
                "premises_nl": [],
                "premises_fol": [],
                "support_idx": [],
                "gold_answer": answer,
                "gold_unit": unit,
                "gold_explanation": str(record.get("cot", "")).strip(),
                "source_path": str(path),
                "id_prefix": prefix,
                "answer_type": answer_type,
                # This label balances both physics topic family and answer style.
                "stratify_label": f"physics::{prefix}::{answer_type}",
            }
        )

    return pd.DataFrame(rows)


logic_df = load_logic_dataset(TYPE1_PATH)
physics_df = load_physics_dataset(TYPE2_PATH)

print("Loaded normalized datasets")
print(f"  Logic rows   : {len(logic_df):,}")
print(f"  Physics rows : {len(physics_df):,}")
print(f"  Total rows   : {len(logic_df) + len(physics_df):,}")
print()
print("Logic question types:")
display(logic_df["question_type"].value_counts().rename_axis("question_type").to_frame("count"))
print("Physics answer types:")
display(physics_df["answer_type"].value_counts().rename_axis("answer_type").to_frame("count"))
print("Physics ID prefixes:")
display(physics_df["id_prefix"].value_counts().rename_axis("id_prefix").to_frame("count"))

# Preview a few normalized examples so we can manually verify that fields look
# correct before building prompts and calling the LLM.
display(logic_df.head(2))
display(physics_df.head(2))

Loaded normalized datasets
  Logic rows   : 808
  Physics rows : 1,755
  Total rows   : 2,563

Logic question types:


,count
question_type,
yes_no_unknown,459
multiple_choice,349


Physics answer types:


,count
answer_type,
numeric_with_unit,1210
missing_answer,401
numeric_no_unit,73
conceptual,71


Physics ID prefixes:


,count
id_prefix,
QA,401
LD,399
CH,290
NL,190
TD,177
DDT,130
THCB,80
DT,68
CHLT,20


,id,group_id,task_type,question_type,question,premises_nl,premises_fol,support_idx,gold_answer,gold_unit,gold_explanation,source_path,stratify_label
0,logic_0000_00,logic_0000,logic,multiple_choice,Which conclusion follows with the fewest premi...,"[If a Python code is well-tested, then the pro...","[∀x (WT(x) → O(x)), ∀x (¬PEP8(x) → ¬WT(x)), ∀x...",[1],Unknown,,Premise 1 states that if a Python project is w...,/home/phuckhang/MyWorkspace/Exact2026/src/data...,logic::multiple_choice
1,logic_0000_01,logic_0000,logic,yes_no_unknown,Does it follow that if all Python projects are...,"[If a Python code is well-tested, then the pro...","[∀x (WT(x) → O(x)), ∀x (¬PEP8(x) → ¬WT(x)), ∀x...","[7, 10]",Yes,,Premise 10 confirms all Python projects are we...,/home/phuckhang/MyWorkspace/Exact2026/src/data...,logic::yes_no_unknown


,id,group_id,task_type,question_type,question,premises_nl,premises_fol,support_idx,gold_answer,gold_unit,gold_explanation,source_path,id_prefix,answer_type,stratify_label
0,physics_TD401,physics_TD401,physics,physics,Calculate the energy stored in capacitor C whe...,[],[],[],45,J,Step 1: Identify the given values for capacita...,/home/phuckhang/MyWorkspace/Exact2026/src/data...,TD,numeric_with_unit,physics::TD::numeric_with_unit
1,physics_TD402,physics_TD402,physics,physics,"Calculate the capacitance C of the capacitor, ...",[],[],[],100,μF,Step 1: Identify the given values from the que...,/home/phuckhang/MyWorkspace/Exact2026/src/data...,TD,numeric_with_unit,physics::TD::numeric_with_unit


# **3. Train / Dev / Test Split**

For this project, use **70% train / 15% dev / 15% test**.

Why this is the best default here:

- **Train 70%**: enough examples for prompt design, formula-template mining, error taxonomy, and later LoRA/fine-tuning if needed.
- **Dev 15%**: enough examples for fast iteration while building the zero-shot, symbolic, and neuro-symbolic pipelines.
- **Test 15%**: enough examples for a final honest estimate without wasting too much limited challenge data.

How the split affects pipeline performance:

- If the **dev set is too small**, you may tune prompts/templates to noise and make bad decisions.
- If the **test set is used during development**, your reported result becomes optimistic and weak for the paper.
- If the **train set is too small**, later fine-tuning or template mining may miss important physics and logic patterns.
- If Type 1 is split by individual questions instead of premise groups, the same premise set can appear in both train and test. This causes leakage and makes the logic pipeline look stronger than it really is.
- For Type 2, stratifying by ID prefix helps keep physics topic families distributed across splits, so one split does not accidentally contain most of one problem type.

Rule for this notebook:

> Use the **dev split** while improving prompts and code. Touch the **test split** only when reporting the baseline result.

In [18]:
# ----------------------------------------------------------------------------
# Create reproducible train / dev / test splits
# ----------------------------------------------------------------------------
# Recommended ratio for this project: 70% train, 15% dev, 15% test.
#
# Why this ratio?
# - 70% train keeps enough examples for prompt design, template mining, and later
#   fine-tuning experiments.
# - 15% dev is large enough for fast iteration while building the pipeline.
# - 15% test is kept untouched until we want an honest estimate of performance.
#
# Important: for Type 1, multiple questions can share the same premise set. If we
# split individual questions randomly, the model/pipeline may see the same
# premises in train and test, which causes leakage. Therefore we split Type 1 by
# `group_id`, where each group is one original premise set.
#
# For Type 2, each row is an independent physics problem. We still stratify by
# `stratify_label` so that formula/topic-like ID families are distributed across
# train/dev/test.
SPLIT_RATIOS = {
    "train": 0.70,
    "dev": 0.15,
    "test": 0.15,
}


def assign_group_splits(
    df: pd.DataFrame,
    group_col: str,
    stratify_col: str,
    ratios: dict[str, float],
    seed: int,
) -> pd.DataFrame:
    """Assign train/dev/test labels while keeping every group in only one split."""
    assert abs(sum(ratios.values()) - 1.0) < 1e-9, "Split ratios must sum to 1.0"

    rng = random.Random(seed)
    split_by_group: dict[str, str] = {}

    # Work at group level, not row level. This prevents Type 1 premise leakage.
    group_table = (
        df[[group_col, stratify_col]]
        .drop_duplicates(subset=[group_col])
        .reset_index(drop=True)
    )

    # Stratify groups by task-specific labels. This keeps answer/topic distribution
    # more stable across train/dev/test than pure random splitting.
    for _, label_groups in group_table.groupby(stratify_col):
        groups = label_groups[group_col].tolist()
        rng.shuffle(groups)

        n = len(groups)
        n_train = int(round(n * ratios["train"]))
        n_dev = int(round(n * ratios["dev"]))

        # Ensure all groups are assigned even when rounding is imperfect.
        train_groups = groups[:n_train]
        dev_groups = groups[n_train : n_train + n_dev]
        test_groups = groups[n_train + n_dev :]

        for group in train_groups:
            split_by_group[group] = "train"
        for group in dev_groups:
            split_by_group[group] = "dev"
        for group in test_groups:
            split_by_group[group] = "test"

    output = df.copy()
    output["split"] = output[group_col].map(split_by_group)
    return output


# Logic split: grouped by original premise set to prevent leakage.
logic_df = assign_group_splits(
    logic_df,
    group_col="group_id",
    stratify_col="stratify_label",
    ratios=SPLIT_RATIOS,
    seed=SEED,
)

# Physics split: each physics row is independent, so group_id is just its own ID.
physics_df = assign_group_splits(
    physics_df,
    group_col="group_id",
    stratify_col="stratify_label",
    ratios=SPLIT_RATIOS,
    seed=SEED,
)

# Combine the two datasets after splitting. Keeping a single table makes later
# zero-shot prompting/evaluation easier, while `task_type` still lets us separate
# logic and physics metrics.
all_df = pd.concat([logic_df, physics_df], ignore_index=True)

# Save the normalized dataset and split file so every future baseline uses the
# same examples. This is important for fair comparisons in the paper.
NORMALIZED_DATA_PATH = ARTIFACTS_DIR / "normalized_dataset.jsonl"
SPLIT_FILE = SPLITS_DIR / f"seed{SEED}_grouped_70_15_15.json"

all_df.to_json(NORMALIZED_DATA_PATH, orient="records", lines=True, force_ascii=False)

split_payload = {
    "seed": SEED,
    "ratios": SPLIT_RATIOS,
    "notes": [
        "Type 1 is split by original premise-set group_id to avoid premise leakage.",
        "Type 2 is stratified by ID prefix / answer type label.",
        "Test split should remain untouched until final baseline reporting.",
    ],
    "ids": {
        split_name: all_df.loc[all_df["split"] == split_name, "id"].tolist()
        for split_name in ["train", "dev", "test"]
    },
}
SPLIT_FILE.write_text(json.dumps(split_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Saved normalized dataset to: {NORMALIZED_DATA_PATH}")
print(f"Saved split file to: {SPLIT_FILE}")
print()
print("Overall split counts:")
display(pd.crosstab(all_df["task_type"], all_df["split"], margins=True))
print("\nLogic split by question type:")
display(pd.crosstab(logic_df["question_type"], logic_df["split"], margins=True))
print("\nPhysics split by answer type:")
display(pd.crosstab(physics_df["answer_type"], physics_df["split"], margins=True))
print("\nPhysics split by ID prefix:")
display(pd.crosstab(physics_df["id_prefix"], physics_df["split"], margins=True))

Saved normalized dataset to: /home/phuckhang/MyWorkspace/Exact2026/artifacts/normalized_dataset.jsonl
Saved split file to: /home/phuckhang/MyWorkspace/Exact2026/artifacts/splits/seed42_grouped_70_15_15.json

Overall split counts:


split,dev,test,train,All
task_type,,,,
logic,119,124,565,808
physics,264,262,1229,1755
All,383,386,1794,2563



Logic split by question type:


split,dev,test,train,All
question_type,,,,
multiple_choice,53,56,240,349
yes_no_unknown,66,68,325,459
All,119,124,565,808



Physics split by answer type:


split,dev,test,train,All
answer_type,,,,
conceptual,11,10,50,71
missing_answer,60,60,281,401
numeric_no_unit,12,10,51,73
numeric_with_unit,181,182,847,1210
All,264,262,1229,1755



Physics split by ID prefix:


split,dev,test,train,All
id_prefix,,,,
CH,44,43,203,290
CHLT,3,3,14,20
DDT,20,19,91,130
DT,10,10,48,68
LD,60,60,279,399
NL,29,28,133,190
QA,60,60,281,401
TD,26,27,124,177
THCB,12,12,56,80


# **4. Zero-Shot Prompt Templates**

A zero-shot baseline means the model receives **only instructions and the current example**. It does not receive solved examples inside the prompt.

This section creates separate prompts for the two official task types:

1. **Logic / Type 1**: the model receives natural-language premises and one question.
2. **Physics / Type 2**: the model receives one physics problem.

For evaluation, the most important rule is that the model must return a predictable structure. Therefore, every prompt asks for a JSON object with these fields:

- `answer`: the final answer;
- `unit`: the physical unit if the problem has one, otherwise an empty string;
- `explanation`: a concise natural-language justification.

This does not force the model to reason correctly, but it makes the baseline easier to parse, evaluate, and compare against later symbolic or neuro-symbolic pipelines.

In [19]:
def format_premises(premises: list[str]) -> str:
    """Convert a list of premises into a numbered text block for the prompt."""
    if not premises:
        return "No premises are provided."

    # Numbering the premises makes the prompt easier to read and helps the model
    # refer to specific evidence in its explanation.
    return "\n".join(f"{index}. {premise}" for index, premise in enumerate(premises, start=1))


def build_logic_prompt(row: pd.Series) -> str:
    """Build the zero-shot prompt for a Type 1 logic example."""
    premises_text = format_premises(row["premises_nl"])

    # The instruction says to answer only from the premises because Type 1 is an
    # entailment-style task. We do not want the model to use outside knowledge.
    return f"""You are solving an educational logic question.
Use only the provided premises. Do not use outside knowledge.

Your task:
1. Decide the final answer to the question.
2. Explain briefly which premise relationships support the answer.
3. Return only valid JSON.

Allowed answer style:
- If the question is yes/no/unknown, use one of: "Yes", "No", "Unknown".
- If the question is multiple-choice, use the option letter such as "A", "B", "C", or "D".

Required JSON format:
{{
  "answer": "...",
  "unit": "",
  "explanation": "..."
}}

Premises:
{premises_text}

Question:
{row["question"]}
"""


def build_physics_prompt(row: pd.Series) -> str:
    """Build the zero-shot prompt for a Type 2 physics example."""
    # The instruction asks for unit separately so that later evaluation can check
    # answer correctness and unit correctness independently.
    return f"""You are solving a physics word problem.

Your task:
1. Identify the relevant physical quantities.
2. Compute or infer the final answer.
3. Return only valid JSON.

Required JSON format:
{{
  "answer": "...",
  "unit": "...",
  "explanation": "..."
}}

Rules:
- Put only the final numeric or conceptual answer in "answer".
- Put only the unit in "unit". If there is no unit, use an empty string.
- Keep the explanation short but clear enough for a student to verify.

Question:
{row["question"]}
"""


def build_zero_shot_prompt(row: pd.Series) -> str:
    """Route one normalized row to the correct zero-shot prompt template."""
    if row["task_type"] == "logic":
        return build_logic_prompt(row)
    if row["task_type"] == "physics":
        return build_physics_prompt(row)
    raise ValueError(f"Unsupported task_type: {row['task_type']}")


# Preview one prompt from each task type before running inference. Manual prompt
# inspection is important because small formatting mistakes can strongly affect
# zero-shot performance.
logic_preview = all_df[all_df["task_type"] == "logic"].iloc[0]
physics_preview = all_df[all_df["task_type"] == "physics"].iloc[0]

print("Logic prompt preview:")
print(build_zero_shot_prompt(logic_preview)[:2_000])
print("\n" + "=" * 80 + "\n")
print("Physics prompt preview:")
print(build_zero_shot_prompt(physics_preview)[:2_000])

Logic prompt preview:
You are solving an educational logic question.
Use only the provided premises. Do not use outside knowledge.

Your task:
1. Decide the final answer to the question.
2. Explain briefly which premise relationships support the answer.
3. Return only valid JSON.

Allowed answer style:
- If the question is yes/no/unknown, use one of: "Yes", "No", "Unknown".
- If the question is multiple-choice, use the option letter such as "A", "B", "C", or "D".

Required JSON format:
{
  "answer": "...",
  "unit": "",
  "explanation": "..."
}

Premises:
1. If a Python code is well-tested, then the project is optimized.
2. If a Python code does not follow PEP 8 standards, then it is not well-tested.
3. All Python projects are easy to maintain.
4. All Python code is well-tested.
5. If a Python code follows PEP 8 standards, then it is easy to maintain.
6. If a Python code is well-tested, then it follows PEP 8 standards.
7. If a Python project is well-structured, then it is optimized.
8.

# **5. LLM Generation Function**

This section defines one function that sends a prompt to the model.

The notebook supports three modes:

1. **Mock mode** (`MOCK_LLM=1`): returns a fake JSON answer without loading a model. Use this mode to test prompt creation, parsing, saving, and evaluation quickly.
2. **Ollama mode** (`EXACT_LLM_BACKEND=ollama`): calls a local Ollama server. This is the easiest local option because it can run a quantized model such as `qwen2.5:7b` without installing `torch` and `transformers`.
3. **Hugging Face mode** (`EXACT_LLM_BACKEND=huggingface`): loads the model directly with `transformers`. Use this later if you want more control over batching, logits, or GPU placement.

For our current machine, Ollama mode is the recommended local 8B-class test path. The model `qwen2.5:7b` has already been pulled locally and fits the challenge parameter constraint.

In [20]:
# The Hugging Face model and tokenizer are global variables so they are loaded
# only once. Ollama mode does not use these variables.
tokenizer = None
model = None


def mock_generate(prompt: str) -> str:
    """Return a fake JSON response for testing the notebook without a real LLM."""
    # The mock output is intentionally simple. Its job is not to be correct; its
    # job is to verify that every downstream step works before we pay model cost.
    return json.dumps(
        {
            "answer": "Unknown",
            "unit": "",
            "explanation": "Mock response used to test the zero-shot pipeline flow.",
        },
        ensure_ascii=False,
    )


def generate_with_ollama(prompt: str) -> str:
    """Generate one raw model response through the local Ollama HTTP API."""
    payload = {
        "model": OLLAMA_MODEL_ID,
        "prompt": prompt,
        "stream": False,
        # JSON mode asks Ollama to constrain output toward a valid JSON object.
        # This strongly improves parse rate for the baseline notebook.
        "format": "json",
        "options": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "num_predict": MAX_NEW_TOKENS,
        },
    }
    request = urllib.request.Request(
        OLLAMA_GENERATE_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )

    try:
        with urllib.request.urlopen(request, timeout=300) as response:
            response_payload = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError(
            "Could not reach Ollama. Start it with: /home/phuckhang/ollama/bin/ollama serve"
        ) from error

    return str(response_payload.get("response", "")).strip()


def load_local_llm():
    """Load the tokenizer and model for local Hugging Face generation."""
    global tokenizer, model

    if tokenizer is not None and model is not None:
        return tokenizer, model

    # Import transformers only when needed. This keeps mock/Ollama mode
    # lightweight and avoids requiring a model stack just to test the pipeline.
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except ImportError as error:
        raise ImportError(
            "Hugging Face mode requires transformers and torch. Use EXACT_LLM_BACKEND=ollama or MOCK_LLM=1."
        ) from error

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

    # device_map="auto" lets transformers place the model on available GPU/CPU.
    # torch_dtype="auto" chooses the dtype recommended by the checkpoint.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    return tokenizer, model


def generate_with_huggingface(prompt: str) -> str:
    """Generate one raw model response from a prompt using transformers."""
    local_tokenizer, local_model = load_local_llm()

    # Chat-tuned models usually perform better when their chat template is used.
    # If the tokenizer does not define a chat template, we fall back to plain text.
    messages = [{"role": "user", "content": prompt}]
    if getattr(local_tokenizer, "chat_template", None):
        model_input_text = local_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        model_input_text = prompt

    inputs = local_tokenizer(model_input_text, return_tensors="pt").to(local_model.device)

    # temperature=0 means greedy decoding for many models. Some transformer
    # versions warn if sampling parameters are passed with do_sample=False, so we
    # keep generation arguments explicit and minimal.
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": TEMPERATURE > 0,
        "pad_token_id": local_tokenizer.eos_token_id,
    }
    if TEMPERATURE > 0:
        generation_kwargs["temperature"] = TEMPERATURE
        generation_kwargs["top_p"] = TOP_P

    outputs = local_model.generate(**inputs, **generation_kwargs)

    # Decode only the newly generated tokens, not the prompt tokens.
    generated_token_ids = outputs[0][inputs["input_ids"].shape[-1] :]
    return local_tokenizer.decode(generated_token_ids, skip_special_tokens=True).strip()


def generate_zero_shot_response(prompt: str) -> str:
    """Generate a raw response using mock mode, Ollama, or Hugging Face."""
    if MOCK_LLM:
        return mock_generate(prompt)
    if LLM_BACKEND == "ollama":
        return generate_with_ollama(prompt)
    if LLM_BACKEND == "huggingface":
        return generate_with_huggingface(prompt)
    raise ValueError(f"Unsupported LLM_BACKEND: {LLM_BACKEND}")


print("Generation mode:", "mock" if MOCK_LLM else LLM_BACKEND)
print("Active model:", "mock" if MOCK_LLM else (OLLAMA_MODEL_ID if LLM_BACKEND == "ollama" else MODEL_ID))

Generation mode: ollama
Active model: qwen2.5:7b


# **6. Parse Model Output**

LLMs do not always follow the requested JSON format. A baseline notebook should not crash when one output is malformed.

This section defines parsing utilities that:

1. try to find a JSON object inside the raw model response;
2. extract `answer`, `unit`, and `explanation`;
3. record whether parsing succeeded;
4. keep the raw response for later error analysis.

The parse success rate is itself an important metric. If a model often fails to produce valid JSON, that model is harder to use in a reliable educational QA system.

In [21]:
def extract_json_object(text: str) -> str | None:
    """Extract the first balanced JSON object substring from a raw model response."""
    if not text:
        return None

    start = text.find("{")
    if start == -1:
        return None

    # Track braces so responses like 'Here is JSON: {...}' can still be parsed.
    # This is more robust than slicing from the first '{' to the last '}'.
    depth = 0
    for position in range(start, len(text)):
        char = text[position]
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start : position + 1]

    return None


def normalize_text(value) -> str:
    """Convert a parsed value to a clean string for evaluation and saving."""
    if value is None:
        return ""
    return str(value).strip()


def parse_model_response(raw_response: str) -> dict:
    """Parse a raw model response into answer, unit, explanation, and status fields."""
    json_text = extract_json_object(raw_response)
    if json_text is None:
        return {
            "pred_answer": "",
            "pred_unit": "",
            "pred_explanation": "",
            "parse_ok": False,
            "parse_error": "No JSON object found",
        }

    try:
        parsed = json.loads(json_text)
    except json.JSONDecodeError as error:
        return {
            "pred_answer": "",
            "pred_unit": "",
            "pred_explanation": "",
            "parse_ok": False,
            "parse_error": str(error),
        }

    # Keep only the fields we need for this baseline. Later pipelines can add
    # optional fields such as fol, cot, premises, confidence, or proof_trace.
    return {
        "pred_answer": normalize_text(parsed.get("answer", "")),
        "pred_unit": normalize_text(parsed.get("unit", "")),
        "pred_explanation": normalize_text(parsed.get("explanation", "")),
        "parse_ok": True,
        "parse_error": "",
    }


# Quick parser smoke tests. These examples check clean JSON, JSON with extra text,
# and a malformed response. They should run fast even without a real model.
parser_examples = [
    '{"answer": "Yes", "unit": "", "explanation": "Premise 1 supports it."}',
    'Here is my answer:\n{"answer": "45", "unit": "J", "explanation": "Use E = 1/2 C V^2."}',
    'I cannot solve this problem.',
]

for example in parser_examples:
    print(parse_model_response(example))

{'pred_answer': 'Yes', 'pred_unit': '', 'pred_explanation': 'Premise 1 supports it.', 'parse_ok': True, 'parse_error': ''}
{'pred_answer': '45', 'pred_unit': 'J', 'pred_explanation': 'Use E = 1/2 C V^2.', 'parse_ok': True, 'parse_error': ''}
{'pred_answer': '', 'pred_unit': '', 'pred_explanation': '', 'parse_ok': False, 'parse_error': 'No JSON object found'}


# **7. Run Zero-Shot Predictions on the Dev Split**

We run the baseline on the **dev split first**, not the test split.

Why dev first:

- dev is for debugging prompts, parsing, and pipeline design;
- test should stay untouched until final reporting;
- if we repeatedly check test performance, the test set becomes part of development and the paper result becomes less trustworthy.

For quick experiments, set `MAX_DEV_EXAMPLES` to a small number such as `20`. For a full dev baseline, set it to `None`.

In [22]:
def run_zero_shot_example(row: pd.Series) -> dict:
    """Run prompt building, generation, and parsing for one normalized example."""
    prompt = build_zero_shot_prompt(row)
    raw_response = generate_zero_shot_response(prompt)
    parsed_response = parse_model_response(raw_response)

    # Save both gold fields and predicted fields. This makes each prediction row
    # self-contained for later inspection, error analysis, and paper tables.
    return {
        "id": row["id"],
        "split": row["split"],
        "task_type": row["task_type"],
        "question_type": row.get("question_type", ""),
        "question": row["question"],
        "gold_answer": row["gold_answer"],
        "gold_unit": row["gold_unit"],
        "gold_explanation": row["gold_explanation"],
        "pred_answer": parsed_response["pred_answer"],
        "pred_unit": parsed_response["pred_unit"],
        "pred_explanation": parsed_response["pred_explanation"],
        "parse_ok": parsed_response["parse_ok"],
        "parse_error": parsed_response["parse_error"],
        "raw_response": raw_response,
        "prompt": prompt,
    }


# Set this to None when you want to run the full dev split. During development,
# a small number keeps iteration fast and avoids accidentally launching hundreds
# of expensive generations.
MAX_DEV_EXAMPLES = 20

# Keep the dev split order stable for reproducibility. We sample only by taking
# the first rows, not by random choice, so repeated runs compare the same subset.
dev_df = all_df[all_df["split"] == "dev"].reset_index(drop=True)
if MAX_DEV_EXAMPLES is not None:
    dev_run_df = dev_df.head(MAX_DEV_EXAMPLES).copy()
else:
    dev_run_df = dev_df.copy()

active_model_name = "mock" if MOCK_LLM else (OLLAMA_MODEL_ID if LLM_BACKEND == "ollama" else MODEL_ID)

print(f"Dev examples available : {len(dev_df):,}")
print(f"Dev examples to run    : {len(dev_run_df):,}")
print(f"Generation backend     : {'mock' if MOCK_LLM else LLM_BACKEND}")
print(f"Active model           : {active_model_name}")

prediction_rows = []
for _, row in tqdm(dev_run_df.iterrows(), total=len(dev_run_df), desc="Zero-shot dev predictions"):
    prediction_rows.append(run_zero_shot_example(row))

predictions_df = pd.DataFrame(prediction_rows)

# Save predictions immediately so partial results are not lost if a later cell
# fails. JSONL is convenient because each line is one complete prediction record.
PREDICTION_FILE = PREDICTIONS_DIR / f"{RUN_ID}_dev_predictions.jsonl"
predictions_df.to_json(PREDICTION_FILE, orient="records", lines=True, force_ascii=False)

print(f"Saved dev predictions to: {PREDICTION_FILE}")
display(predictions_df.head())

Dev examples available : 383
Dev examples to run    : 20
Generation backend     : ollama
Active model           : qwen2.5:7b


Zero-shot dev predictions: 100%|██████████| 20/20 [17:48<00:00, 53.42s/it]

Saved dev predictions to: /home/phuckhang/MyWorkspace/Exact2026/artifacts/predictions/B01_zero_shot_20260512_211834_dev_predictions.jsonl


,id,split,task_type,question_type,question,gold_answer,gold_unit,gold_explanation,pred_answer,pred_unit,pred_explanation,parse_ok,parse_error,raw_response,prompt
0,logic_0008_00,dev,logic,multiple_choice,"Based on the premises, what can we conclude ab...",B,,Premise 4 and premise 3 confirm the curriculum...,B,,Premises 3 and 4 indicate that the curriculum ...,True,,"{\n ""answer"": ""B"",\n ""unit"": """",\n ""explana...",You are solving an educational logic question....
1,logic_0008_01,dev,logic,yes_no_unknown,Does the combination of faculty priorities and...,Yes,,Faculty priorities in premise 4 (pedagogical t...,Yes,,"According to premise 4, if a faculty prioritiz...",True,,"{\n ""answer"": ""Yes"",\n ""unit"": """",\n ""expla...",You are solving an educational logic question....
2,logic_0014_00,dev,logic,multiple_choice,"Based on Dr. John's qualifications, which stat...",Unknown,,"Premise 7 states Dr. John has a PhD, premise 3...",A,,Dr. John can teach undergraduate courses becau...,True,,"{\n ""answer"": ""A"",\n ""unit"": """",\n ""explana...",You are solving an educational logic question....
3,logic_0014_01,dev,logic,yes_no_unknown,"Can Dr. John teach undergraduate courses, acco...",No,,"Premise 7 confirms Dr. John has a PhD, premise...",Yes,,"According to premise 2, lecturers with a degre...",True,,"{\n ""answer"": ""Yes"",\n ""unit"": """",\n ""expla...",You are solving an educational logic question....
4,logic_0019_00,dev,logic,yes_no_unknown,Do all employees receive a bonus?,Yes,,Premise 6 states everyone is eligible for a pr...,Yes,,"From premise 4, we know every person has a res...",True,,"{\n ""answer"": ""Yes"",\n ""unit"": """",\n ""expla...",You are solving an educational logic question....


# **8. Basic Baseline Evaluation**

This section computes simple automatic metrics for the zero-shot baseline.

The metrics are intentionally simple because this is the first baseline:

- **parse rate**: how often the model produced valid JSON;
- **answer exact match**: whether the predicted answer string exactly matches the gold answer after normalization;
- **unit exact match**: for physics examples, whether the predicted unit exactly matches the gold unit;
- **metrics by task type**: logic and physics should be analyzed separately because they require different reasoning skills.

These metrics are not the final challenge metric, but they are enough to tell whether the baseline is working and where it fails.

In [24]:
def normalize_for_exact_match(value: str) -> str:
    """Normalize text before exact-match comparison."""
    # This keeps evaluation simple and transparent. Later we can add better
    # numeric tolerance, unit conversion, and symbolic equivalence checks.
    return str(value or "").strip().lower()


def add_basic_correctness_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add exact-match correctness columns to a prediction dataframe."""
    output = df.copy()

    output["answer_exact_match"] = (
        output["pred_answer"].map(normalize_for_exact_match)
        == output["gold_answer"].map(normalize_for_exact_match)
    )

    # Convert to pandas nullable boolean dtype before assigning pd.NA. A normal
    # numpy bool column can store only True/False, so assigning pd.NA to logic
    # rows may raise LossySetitemError / Invalid value 'nan' for dtype 'bool'.
    output["unit_exact_match"] = (
        output["pred_unit"].map(normalize_for_exact_match)
        == output["gold_unit"].map(normalize_for_exact_match)
    ).astype("boolean")

    # Unit accuracy is meaningful mainly for physics. For logic, gold_unit is
    # intentionally empty, so unit_exact_match is not a useful quality signal.
    output.loc[output["task_type"] != "physics", "unit_exact_match"] = pd.NA
    return output


def summarize_predictions(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize parse rate, answer accuracy, and physics unit accuracy."""
    rows = []

    for group_name, group_df in [("all", df), *list(df.groupby("task_type"))]:
        metric_row = {
            "group": group_name,
            "n_examples": len(group_df),
            "parse_rate": group_df["parse_ok"].mean(),
            "answer_exact_match": group_df["answer_exact_match"].mean(),
        }

        physics_or_all = group_df[group_df["task_type"] == "physics"]
        if len(physics_or_all) > 0:
            metric_row["physics_unit_exact_match"] = physics_or_all["unit_exact_match"].mean()
        else:
            metric_row["physics_unit_exact_match"] = pd.NA

        rows.append(metric_row)

    return pd.DataFrame(rows)


predictions_df = add_basic_correctness_columns(predictions_df)
metrics_df = summarize_predictions(predictions_df)
active_model_name = "mock" if MOCK_LLM else (OLLAMA_MODEL_ID if LLM_BACKEND == "ollama" else MODEL_ID)

REPORT_FILE = REPORTS_DIR / f"{RUN_ID}_dev_metrics.json"
metrics_payload = {
    "run_id": RUN_ID,
    "model_id": active_model_name,
    "mock_llm": MOCK_LLM,
    "llm_backend": "mock" if MOCK_LLM else LLM_BACKEND,
    "max_dev_examples": MAX_DEV_EXAMPLES,
    "prediction_file": str(PREDICTION_FILE),
    "metrics": metrics_df.to_dict(orient="records"),
}
REPORT_FILE.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Saved dev metrics to: {REPORT_FILE}")
display(metrics_df)

print("Examples with wrong answer or parse failure:")
error_view = predictions_df[(~predictions_df["answer_exact_match"]) | (~predictions_df["parse_ok"])]
display(
    error_view[
        [
            "id",
            "task_type",
            "gold_answer",
            "gold_unit",
            "pred_answer",
            "pred_unit",
            "parse_ok",
            "parse_error",
        ]
    ].head(20)
)

Saved dev metrics to: /home/phuckhang/MyWorkspace/Exact2026/artifacts/reports/B01_zero_shot_20260512_211834_dev_metrics.json


,group,n_examples,parse_rate,answer_exact_match,physics_unit_exact_match
0,all,20,1.0,0.35,<NA>
1,logic,20,1.0,0.35,<NA>


Examples with wrong answer or parse failure:


,id,task_type,gold_answer,gold_unit,pred_answer,pred_unit,parse_ok,parse_error
2,logic_0014_00,logic,Unknown,,A,,True,
3,logic_0014_01,logic,No,,Yes,,True,
6,logic_0025_00,logic,Unknown,,A,,True,
7,logic_0025_01,logic,Unknown,,B,,True,
8,logic_0033_00,logic,No,,Yes,,True,
10,logic_0043_00,logic,Unknown,,A,,True,
11,logic_0043_01,logic,No,,Yes,,True,
12,logic_0044_00,logic,Unknown,,B,,True,
13,logic_0044_01,logic,No,,Yes,,True,
15,logic_0059_00,logic,No,,D,,True,
